# 05 · Prediction & Model Export
This notebook independently loads the training bundle made by notebook 03, demonstrates predictions on held-out records, chooses by PR-AUC metrics from notebook 04, refits the selected pipeline on all labeled examples, and exports model and metadata. Run notebooks 03 and 04 first. Nothing is deployed or connected to the website.


In [ ]:
%pip -q install pandas numpy scikit-learn joblib
from pathlib import Path
import pandas as pd, numpy as np, joblib, json, datetime
from sklearn.metrics import *
try:
 from google.colab import drive
 drive.mount('/content/drive')
except ImportError: pass
ROOT=Path('/content/drive/MyDrive/insurance_fraud_ml'); bp=ROOT/'models/trained_models.joblib'; mp=ROOT/'results/model_comparison.csv'; dp=ROOT/'data'
if not bp.exists() or not mp.exists(): raise FileNotFoundError('Run notebooks 03 and 04 first')
bundle=joblib.load(bp); models=bundle['models']; Xte=bundle['X_test']; yte=bundle['y_test']; metrics=pd.read_csv(mp); best_name=metrics.sort_values('pr_auc',ascending=False).iloc[0]['model']; model=models[best_name]
sample=Xte.head(10); predictions=model.predict(sample); probabilities=model.predict_proba(sample)[:,1]
demo=pd.DataFrame({'actual':yte.loc[sample.index].values,'predicted':predictions,'fraud_probability':probabilities},index=sample.index); demo['actual_label']=demo.actual.map({0:'Legitimate',1:'Fraud'}); demo['predicted_label']=demo.predicted.map({0:'Legitimate',1:'Fraud'}); display(demo)
print('Probability is the classifier score for class 1, not a guaranteed real-world probability.')

## Export for future backend integration
The bundled split does not include all original rows. Re-upload/reload the original dataset from Drive to refit on all labeled records. The untouched held-out metrics are retained for reporting.

In [ ]:
paths=list(dp.glob('*.csv')) if dp.exists() else []
if not paths: raise FileNotFoundError('Place the unchanged source CSV in Drive project data folder')
csv=next((p for p in paths if p.name not in ['X_train.csv','X_test.csv','y_train.csv','y_test.csv']),None)
if csv is None: raise FileNotFoundError('Could not locate original CSV; upload it to Drive data folder')
df=pd.read_csv(csv); target='FraudFound_P'; df[target]=pd.to_numeric(df[target],errors='coerce'); df=df.dropna(subset=[target]); df[target]=df[target].astype(int)
X=df.drop(columns=[target]); y=df[target]; X=X.drop(columns=[c for c in X if any(k in c.lower() for k in ['policy_number','claim_id','record_id','customer_id'])],errors='ignore')
final=models[best_name]; final.fit(X,y); (ROOT/'models').mkdir(parents=True,exist_ok=True); (ROOT/'results').mkdir(exist_ok=True); joblib.dump(final,ROOT/'models/best_model.joblib')
score=metrics.sort_values('pr_auc',ascending=False).iloc[0].to_dict(); metrics.to_csv(ROOT/'results/model_comparison.csv',index=False); (ROOT/'results/model_metrics.json').write_text(json.dumps(score,indent=2))
metadata={'dataset_name':csv.name,'dataset_source':'https://www.kaggle.com/datasets/shivamb/vehicle-claim-fraud-detection','target_variable':target,'training_date':datetime.date.today().isoformat(),'records':int(len(df)),'training_records':int(len(df)-len(yte)),'testing_records':int(len(yte)),'selected_model':best_name,'selected_features':X.columns.tolist(),'evaluation_metrics':score,'random_state':42,'classification_threshold':0.5}
(ROOT/'models/model_metadata.json').write_text(json.dumps(metadata,indent=2))
est=final.named_steps['model']; names=final.named_steps['prep'].get_feature_names_out(); imp=est.feature_importances_ if hasattr(est,'feature_importances_') else (np.abs(est.coef_[0]) if hasattr(est,'coef_') else None)
if imp is not None: pd.DataFrame({'feature':names,'importance':imp}).sort_values('importance',ascending=False).to_csv(ROOT/'results/feature_importance.csv',index=False)
loaded=joblib.load(ROOT/'models/best_model.joblib'); print('Reloaded model prediction:',loaded.predict(X.head(1)).tolist()); print('Exported artifacts to',ROOT)

Exported: best_model.joblib, model_metadata.json, model_comparison.csv, model_metrics.json and feature_importance.csv where supported. This demonstrates training and serialization; actual outputs are pending until execution on the original data.